<style>body,.jp-Notebook,.jp-NotebookPanel,.jp-RenderedHTMLCommon,.jp-RenderedMarkdown,.jp-Cell,.jp-OutputArea,.jp-OutputArea-output{background:#111!important;color:#eee!important}.jp-RenderedHTMLCommon h1,.jp-RenderedHTMLCommon h2,.jp-RenderedHTMLCommon h3{color:#fff!important}.jp-RenderedHTMLCommon a{color:#7db7ff!important}pre,code,.jp-OutputArea pre,.jp-RenderedText{background:#181818!important;color:#eee!important}</style>

# NLP Text Classification
---
 ### Overview
This project applies Natural Language Processing to classify over 8000 social media comments into Positive and Negative categories. Primarily, it will utilize Pyhtons Sklearn, Numpy packages, Support Vector Machines, and Cross-Validation to perform the sentiment analysis

### Business Problem
Many large companies have massive amounts of data that they have to deal with which can make manual analysis very difficult. For example, consider a company that posts a video and recieves 10,000 reviews, how will they sift through every review in a timely fasion? It could take weeks or months. This project solves this problem utilizing sentiment anotation analysis to categorize data at scale.

> The goal of this project is to classifcy social media comments using machine learning based on if they are positive or negative comments

### Load the data and train the model

In [7]:
import csv
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SelectKBest
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

import numpy as np
np.random.seed(42)
import random
random.seed(42)

# Load the Data
X_txt = []
y = []
with open('./sentiment-twitter-data.tsv') as in_file:
    iCSV = csv.reader(in_file, delimiter='\t')
    for row in iCSV:
        X_txt.append(row[-1])
        y.append(row[-2])
        

X_txt_train, X_txt_test, y_train, y_test = train_test_split(X_txt, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('vec', CountVectorizer()),
    ('clf', LinearSVC(random_state=42, max_iter=10000))
])

params= {
    'vec__stop_words': ['english', None],
    'vec__lowercase': [False, True],
    'vec__min_df': [1, 5, 10],
    'vec__ngram_range': [(1,1), (1,2)],
    'clf__C':[0.01, 0.1,1.0],
}


clf = GridSearchCV(pipe, 
                   params, 
                   cv=5
                  )

clf.fit(X_txt_train, y_train)

preds = clf.predict(X_txt_test)


print('Best score:', clf.best_score_)
print('Best params:', clf.best_params_)
print("Best micro F1 Score:", f1_score(y_test, preds, average='micro'))

Best score: 0.4483688280640125
Best params: {'clf__C': 0.1, 'vec__lowercase': True, 'vec__min_df': 1, 'vec__ngram_range': (1, 1), 'vec__stop_words': None}
Best micro F1 Score: 0.46595877576514677


In [10]:
print("Total text records:", len(X_txt))
print("Total labels:", len(y))


Total text records: 8002
Total labels: 8002


### See results

In [20]:
from collections import Counter

# Distribution of all 8,002 comments
distribution = Counter(y)

print("Comment Distribution:")
for label, count in distribution.items():
    percentage = (count / len(y)) * 100
    print(f"{label}: {count} ({percentage:.2f}%)")



Comment Distribution:
positive: 2974 (37.17%)
negative: 1159 (14.48%)
neutral: 1446 (18.07%)
objective-OR-neutral: 1391 (17.38%)
objective: 1032 (12.90%)


In [13]:
X_txt_train, X_txt_test, y_train, y_test = train_test_split(X_txt, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('vec', CountVectorizer()),
    ('skb', SelectKBest()),
    ('clf', LinearSVC(random_state=42))])

params= {
    'skb__k': [5, 10,1000,2000],
    'vec__ngram_range': [(1,1), (1,2)],
    'clf__C':[0.01, 0.1,1.0],
}


clf = GridSearchCV(pipe, params, cv=5)

clf.fit(X_txt_train, y_train)

preds = clf.predict(X_txt_test)


print('Best score:', clf.best_score_)

Best score: 0.44649334016393444


### Visualize what machine learning model did...

In [21]:
for text, prediction in zip(X_txt_test[:10], preds[:10]):
    print("Comment:", text)
    print("Predicted sentiment:", prediction)
    print()

Comment: My dad reckons if this was in Nigeria the president would have issued a shoot on sight order since Saturday!
Predicted sentiment: negative

Comment: @BrielleWatkins I think you can get them on Ticketek? And they go on sale tomorrow.. I think... :)
Predicted sentiment: neutral

Comment: @sue_bryce hey sue. So excited to see you at WPPI U tomorrow. We're coming don Canada and would love to buy you a drink. @catG_photo
Predicted sentiment: positive

Comment: LMAO RT @RichardKingson: Went looting in Spurs last night, broke into the White Hart Lane cabinet so left with a pile of dust and a Bale DVD
Predicted sentiment: positive

Comment: @BreeKaye08 haha I just may have to see him at rodeo this year. Thankfully it's a Saturday!!!
Predicted sentiment: positive

Comment: 7 of the top 10 Search Engine referrals for TVE today are Mike & Molly related, thanks to Melissa McCarthy's win at the #Emmys!
Predicted sentiment: positive

Comment: Jersey shore season four tonight :) x
Predicted 

# Results
---
### Test Results: 
The model came back with cross-validation result of .446 meaning that 44.64% of the time the model correctly predicts a comments category.

- about ~37% of the comments are positive
- about ~14% of the comments are negative
- the remainder of the comments are nuetral or objective

